# Systolic-Array DNN Accelerator Simulator — Results & Validation
### Module scope: stationary schemes (OS/WS/IS) · memory layouts (Row/Column/Channel-major) · casting schemes (Multicast/Unicast/Hybrid) · analytical configuration-chooser

This notebook loads **real result files** produced by this project's Verilator + cocotb RTL verification suite and its Python analytical cost model, and presents them as tables and charts suitable for a live evaluation and for inclusion in a thesis.

**Every value below is labelled `measured (RTL)` or `model`.** `measured (RTL)` means the number came directly from a cycle-accurate Verilator simulation of the real hardware description, driven by a cocotb testbench, checked against a TensorFlow golden reference. `model` means the number came from the analytical Python cost formulas — computed instantly, with no hardware simulation. Nothing on this page is invented; every cell reads from a CSV/JSON file that ships in `data_bundle.zip`.

---
## How to run this notebook
1. Open **[colab.research.google.com](https://colab.research.google.com)** and upload this `.ipynb` file (File → Upload notebook), *or* open it directly from GitHub if the repo is pushed there.
2. Run the first code cell below. It will prompt you to **upload `data_bundle.zip`** — select the file from `results/thesis_notebook/data_bundle.zip` in this project.
3. Run all remaining cells top to bottom (Runtime → Run all). No installs are needed — everything used (`pandas`, `matplotlib`, `numpy`) is preinstalled in Colab.
4. For the thesis: right-click any chart → *Save image as*, or select a table cell's output and copy it — `pandas` tables also export directly with `df.to_latex()` if your thesis is in LaTeX (shown at the end).

In [ ]:
# ------------------------------------------------------------------
# Setup — upload data_bundle.zip when prompted (skip prompt if files
# are already present, e.g. when re-running locally with Jupyter).
# ------------------------------------------------------------------
import os, zipfile, sys
from pathlib import Path

BUNDLE_DIR = Path('data_bundle')
if not BUNDLE_DIR.exists():
    try:
        from google.colab import files
        print('Please select data_bundle.zip (from results/thesis_notebook/ in the project repo):')
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name) as z:
            z.extractall('.')
    except ImportError:
        # Not running in Colab -- look for a local copy next to this notebook.
        local_zip = Path('data_bundle.zip')
        assert local_zip.exists(), 'Place data_bundle.zip next to this notebook, or run in Colab.'
        with zipfile.ZipFile(local_zip) as z:
            z.extractall('.')

assert BUNDLE_DIR.exists(), 'data_bundle/ not found after extraction.'
print('Data bundle ready:', sorted(p.name for p in BUNDLE_DIR.iterdir()))


In [ ]:
import json, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': True,
                      'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})

# Consistent colour roles used throughout this notebook and this project's
# other results pages: teal = measured on real RTL, violet = analytical model.
MEASURED = '#0C7A73'
MODEL    = '#6B4FA0'
GAP      = '#B24A18'   # used only where model and measured genuinely differ
DF_COLOR = {'OS': '#2a78d6', 'IS': '#eb6834', 'WS': '#1baf7a'}

GC_RAW = BUNDLE_DIR / 'golden_check' / 'raw'
GC_FIG = BUNDLE_DIR / 'golden_check' / 'figures'
CHOOSER = BUNDLE_DIR / 'chooser'
EDGE_CLOUD = BUNDLE_DIR / 'edge_cloud_sample'

def source_tag(text, kind='measured'):
    """Small coloured badge so every printed table states its own provenance."""
    color = MEASURED if kind == 'measured' else MODEL
    label = 'MEASURED (RTL)' if kind == 'measured' else 'MODEL (analytical)'
    print(f'\033[1m[{label}]\033[0m {text}')


---
## 0 · What this module covers

This notebook documents the **stationary-scheme / memory-layout / casting-scheme** hardware knobs and the **analytical configuration-chooser** built on top of them.

| Knob | Options | What it controls |
|---|---|---|
| **Stationary scheme (dataflow)** | Output-Stationary (OS), Weight-Stationary (WS), Input-Stationary (IS) | Which operand stays resident in the PE array while the others stream through |
| **Memory layout** | Row-major, Column-major, Channel-major | The order tensors are laid out in off-chip DRAM, which changes how well consecutive fetches coalesce into AXI bursts |
| **Casting scheme** | Multicast, Unicast, Hybrid | How a value shared by multiple PEs is fetched off-chip: once (multicast), once per consuming PE (unicast), or split by operand (hybrid) |
| **Configuration chooser** | — | An analytical tool that scores all 3×3×3 = 27 combinations for a given workload, array size, and memory budget, and ranks them by a chosen goal (off-chip traffic / latency / energy / a weighted mix) — without running RTL per query |

The sections below prove, with real numbers: **(1)** the RTL implementing these knobs is functionally correct against a TensorFlow golden reference, **(2)** the traffic each knob produces is measured and understood, and **(3)** the chooser's analytical model, built on top of that measured behaviour, makes decisions that agree with it — **including the places where it does *not* agree exactly**, which are shown just as clearly as the places where it does.

---
## 1 · Simulator correctness vs. the TensorFlow golden reference

Every RTL configuration below was run through the **same procedure**: Verilator compiles the SystemVerilog design into a cycle-accurate simulator; a cocotb Python testbench drives it with a real DNN layer's weights and inputs; the output is compared element-by-element against an independently computed TensorFlow result (the *golden reference*); a run passes if every element is within **5% of the layer's largest value** (a tolerance sized to absorb expected fixed-point quantization, not to hide real errors).

**Reading the table below:** each row is one RTL configuration run against its
golden reference. Every column now states its own unit, and the measured and
golden values are shown side by side rather than only as a percentage.

| Column | Meaning | Unit |
|---|---|---|
| `Stationary scheme` | which operand is held in the PE array (OS / IS / WS) | — |
| `Memory layout` | how the tensor is packed in DRAM | — |
| `Casting scheme` | off-chip read policy for values shared between PEs | — |
| `Measured value, RTL` | the accelerator's output at the worst-deviating element | activation units |
| `Golden value, TensorFlow` | the reference output at that same element | activation units |
| `Absolute error` | \|measured − golden\| at that element | activation units |
| `Tolerance` | pass/fail threshold, 5% of the layer's full-scale value | activation units |
| `Relative error` | the same error expressed against full scale | % of full scale |
| `Safety margin` | tolerance ÷ error — how far inside the pass line the worst case sits | × (times) |

*Activation units* are the layer's own dimensionless output numbers,
reconstructed from the RTL's fixed-point words (Q-format with
`frac_x + frac_w` = 28 fractional bits).

Both the measured and golden values are recomputed here from the raw RTL
output tensor and the TensorFlow reference, then **cross-checked against the
error each run already recorded in its own verdict file** — all 29
configurations agree exactly, so these columns are reconstructions of the
recorded result, not a re-measurement of it.

In [ ]:
# Enriched correctness table: measured value, golden value and the error
# between them in SEPARATE columns, with units in every header.
# Generated by scripts/make_thesis_tables.py, which recomputes both values
# from the raw RTL output tensor and the TensorFlow reference, then
# cross-checks each row against the error that run already recorded.
detail_all = pd.read_csv(GC_FIG / 'correctness_detail.csv')

UNIT = 'activation units (dimensionless)'
TOL_PCT = 5.0

# The raw results also contain debug probes from the F1/F4 investigations
# (deliberately-broken states used to find those bugs). They are labelled in
# the CSV rather than deleted; the headline table shows validation runs only.
n_probe = (detail_all['Run type'] == 'debug probe').sum()
detail = detail_all[detail_all['Run type'] == 'validation'].reset_index(drop=True)

detail['Safety margin'] = detail['Safety margin (times below tolerance)']
detail['Error (% full scale)'] = detail['Relative error (% of full scale)']

source_tag(f'{len(detail)} validation configurations -- measured vs golden, '
           f'values in {UNIT}', 'measured')
print(f'({n_probe} debug-probe runs are excluded from this table but kept and '
      f'labelled in correctness_detail.csv.)\n')

show = detail[[
    'Configuration', 'Stationary scheme', 'Memory layout', 'Casting scheme',
    f'Measured value, RTL ({UNIT})',
    f'Golden value, TensorFlow ({UNIT})',
    f'Absolute error ({UNIT})',
    f'Tolerance ({UNIT})',
    'Error (% full scale)', 'Safety margin', 'Result',
]].copy()

for c in show.columns:
    if c.startswith(('Measured', 'Golden')):
        show[c] = show[c].round(6)
    elif c.startswith(('Absolute error', 'Tolerance (')):
        show[c] = show[c].map(lambda v: f'{v:.3e}')
show['Error (% full scale)'] = show['Error (% full scale)'].round(4)
show['Safety margin'] = show['Safety margin'].round(0).astype(int).astype(str) + 'x'

# `f2` feeds the two charts below, with readable column names.
f2 = detail.rename(columns={
    'Relative error (% of full scale)': 'max_rel_err_pct',
    'Stationary scheme': 'family',
    'Configuration': 'config'})[['config', 'family', 'max_rel_err_pct']]
f2 = f2.sort_values('max_rel_err_pct', ascending=False).reset_index(drop=True)

show

**Two charts follow, deliberately at two different scales.** The first is scaled against the 5% pass/fail line, which is the number that matters for *correctness* — but on that scale every real error (0.009%–0.05%) looks like it's sitting at zero, so it can't show the differences *between* configurations. The second chart re-scales to the errors' own range so those differences are actually visible.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = f2['family'].map(lambda f: DF_COLOR.get(f, '#9a9690'))
y = np.arange(len(f2))
ax.barh(y, f2['max_rel_err_pct'], color=colors, height=0.62)
ax.axvline(TOL_PCT, color='#d03b3b', linestyle='--', linewidth=1.5, label=f'{TOL_PCT:.0f}% tolerance (fail threshold)')
ax.set_yticks(y)
ax.set_yticklabels(f2['config'], fontsize=7)
ax.set_xlabel('Largest relative error (% of full scale)  --  measured RTL vs TensorFlow golden')
ax.set_xlim(0, TOL_PCT * 1.15)
ax.set_title(f'(1a) Correctness margin, scaled to the tolerance -- {len(f2)} configurations, all PASS, worst case {f2["max_rel_err_pct"].max():.3f}% (100x inside tolerance)')
handles = [plt.Rectangle((0,0),1,1, color=c) for c in DF_COLOR.values()] + [plt.Line2D([0],[0], color='#d03b3b', linestyle='--')]
ax.legend(handles, list(DF_COLOR.keys()) + ['5% tolerance'], loc='lower right', fontsize=8)
fig.tight_layout()
plt.show()
print(f'\nWorst-case error is {TOL_PCT / f2["max_rel_err_pct"].max():.0f}x inside the tolerance band -- '
      f'the residual error is consistent with pure fixed-point quantization, not a functional bug.')


In [ ]:
# Zoomed-in version: same data, x-axis scaled to the errors' own range so the
# real differences BETWEEN configurations are visible (they are invisible on
# the tolerance-scaled chart above, since all 26 sit under 1% of that axis).
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(y, f2['max_rel_err_pct'], color=colors, height=0.62)
ax.set_yticks(y)
ax.set_yticklabels(f2['config'], fontsize=7)
ax.set_xlabel('Largest relative error (% of full scale)')
ax.set_xlim(0, f2['max_rel_err_pct'].max() * 1.12)
for yi, v in zip(y, f2['max_rel_err_pct']):
    ax.text(v, yi, f'  {v:.4f}%', va='center', fontsize=6.5, color='#3a3a38')
ax.set_title('(1b) The same 26 configs, zoomed to the error scale itself -- differences between\nconfigs (0.009% to 0.050%, a 5.6x spread) are now visible')
fig.tight_layout()
plt.show()
print('The largest errors belong to mnist_cnn layer 1 and layer 2 (0.019% and 0.050%) -- both still')
print('~100x and ~26x inside tolerance respectively. The spread reflects each layer\'s own fixed-point')
print('rounding profile, not a difference in correctness.')


---

## Stationary scheme × memory layout × casting scheme

The previous section mixed all configurations together, including the
STAMP/PAGED memory-backend variants. This section isolates the **three
configuration knobs** — stationary scheme, memory layout, casting scheme —
and asks two separate questions about them.

### Is a *correctness* chart across these three axes meaningful?

**Yes, but only as an invariance result — it cannot rank them.** Every
configuration of a given layer records exactly the *same* error (all eight
`mnist_cnn/layer_00` configurations sit at 0.008926% of full scale; all nine
`tiny_cnn/layer_00` configurations at 0.013185%). That is the expected
outcome: the residual error is fixed-point **quantisation** error of the
layer's arithmetic, and these three knobs only change the *order and route*
by which identical operands reach the PEs — never the arithmetic itself.

So the chart below is flat by construction. It is still worth showing,
because flatness is the claim being proved: **these three knobs are pure
performance choices and cost nothing in accuracy.** What it must not be used
for is picking a winner — for that, use the performance table that follows.

In [ ]:
# Correctness across the three configuration axes: expected to be FLAT.
axes_df = pd.read_csv(GC_FIG / 'config_axes_performance.csv')

inv = (axes_df.groupby('Workload / layer')['Relative error (% of full scale)']
       .agg(['nunique', 'min', 'max', 'count'])
       .rename(columns={'nunique': 'Distinct error values',
                        'min': 'Lowest error (% full scale)',
                        'max': 'Highest error (% full scale)',
                        'count': 'Configurations tested'}))
inv = inv[['Configurations tested', 'Distinct error values',
           'Lowest error (% full scale)', 'Highest error (% full scale)']]

source_tag('Correctness invariance across stationary / layout / casting',
           'measured')
print('If "Distinct error values" is 1 for every layer, the three knobs are')
print('provably correctness-neutral on the configurations measured.\n')
inv.round(6)

### What *does* change across these axes: performance

These are the metrics the three knobs actually move, all measured on real
RTL runs. Units are stated in every axis label and column header:

| Metric | Unit | What it means |
|---|---|---|
| Execution time | clock cycles | end-to-end cycles for the layer |
| Off-chip read bursts | AXI AR requests (count) | how many separate DRAM read transactions were issued |
| Off-chip data transferred | AXI beats (count) | how many data words crossed the DRAM interface |

The comparison below holds everything else fixed and varies **one axis at a
time** on `tiny_cnn/layer_00`, so each panel is a controlled experiment.

In [ ]:
# One axis varied at a time, everything else held fixed -- measured RTL.
L = 'tiny_cnn/layer_00'
d = axes_df[axes_df['Workload / layer'] == L]

CYC = 'Execution time (clock cycles)'
AR  = 'Off-chip read bursts (AXI AR requests)'
BEA = 'Off-chip data transferred (AXI beats)'

stationary = d[(d['Memory layout'] == 'CHANNEL_MAJOR') &
               (d['Casting scheme'] == 'MULTICAST') &
               (d['Array size (PEs)'] == '8x8')]
layout = d[(d['Stationary scheme'] == 'OS') &
           (d['Casting scheme'] == 'MULTICAST') &
           (d['Array size (PEs)'] == '8x8')]
casting = d[(d['Stationary scheme'] == 'OS') &
            (d['Memory layout'] == 'CHANNEL_MAJOR') &
            (d['Array size (PEs)'] == '8x8')]

fig, axs = plt.subplots(1, 3, figsize=(14, 4.4))

# (a) stationary scheme -- WS is orders of magnitude slower, so log scale
a = stationary.sort_values(CYC)
axs[0].bar(a['Stationary scheme'], a[CYC],
           color=[DF_COLOR.get(s, MEASURED) for s in a['Stationary scheme']])
axs[0].set_yscale('log')
axs[0].set_ylabel('Execution time (clock cycles, log scale)')
axs[0].set_title('(a) Stationary scheme\nlayout=CHANNEL_MAJOR, casting=MULTICAST, 8x8')
for x, v in zip(range(len(a)), a[CYC]):
    axs[0].text(x, v, f'{v:,}', ha='center', va='bottom', fontsize=8)

# (b) memory layout -- identical beats, very different AR request counts
b = layout.sort_values(CYC)
xb = np.arange(len(b)); w = 0.38
axs[1].bar(xb - w/2, b[CYC], w, label='Execution time (cycles)', color=MEASURED)
axs[1].bar(xb + w/2, b[AR], w, label='Read bursts (AR requests)', color=MODEL)
axs[1].set_xticks(xb)
axs[1].set_xticklabels([l.replace('_MAJOR', '') for l in b['Memory layout']])
axs[1].set_ylabel('Count')
axs[1].set_title('(b) Memory layout\nstationary=OS, casting=MULTICAST, 8x8')
axs[1].legend(fontsize=8)

# (c) casting scheme -- this axis moves the data VOLUME
c = casting.sort_values(BEA)
xc = np.arange(len(c))
axs[2].bar(xc - w/2, c[BEA], w, label='Data transferred (AXI beats)', color=MEASURED)
axs[2].bar(xc + w/2, c[CYC], w, label='Execution time (cycles)', color=GAP)
axs[2].set_xticks(xc)
axs[2].set_xticklabels(c['Casting scheme'], fontsize=8)
axs[2].set_ylabel('Count')
axs[2].set_title('(c) Casting scheme\nstationary=OS, layout=CHANNEL_MAJOR, 8x8')
axs[2].legend(fontsize=8)

for ax in axs:
    ax.tick_params(axis='x', labelsize=9)
fig.suptitle(f'Stationary / layout / casting on {L} -- one axis varied at a time '
             '(measured RTL)', y=1.02)
fig.tight_layout()
plt.show()

print('(a) Stationary: IS is fastest here (5,220 cycles) and WS slowest (153,738)')
print('    -- WS re-fetches weights once per invocation on this layer.')
print('(b) Layout: CHANNEL_MAJOR and ROW/COLUMN_MAJOR move the SAME 1,836 beats,')
print('    but ROW/COLUMN need 1,794 read bursts vs 138 -- burst coalescing, not')
print('    traffic volume, is what makes CHANNEL_MAJOR 1.86x faster.')
print('(c) Casting: this axis changes volume -- MULTICAST 1,836 beats, HYBRID')
print('    11,664, UNICAST 20,736 (11.3x), with execution time tracking it.')

In [ ]:
# The same three axes as a single table, with units in every header.
source_tag(f'Stationary / layout / casting on {L} -- measured RTL', 'measured')

tbl = d[['Stationary scheme', 'Memory layout', 'Casting scheme',
         'Array size (PEs)', CYC, AR, BEA,
         'Relative error (% of full scale)']].copy()
tbl = tbl.rename(columns={'Relative error (% of full scale)':
                          'Error (% full scale)'})
tbl['Memory layout'] = tbl['Memory layout'].str.replace('_MAJOR', '', regex=False)
tbl = tbl.sort_values(CYC).reset_index(drop=True)
tbl['Error (% full scale)'] = tbl['Error (% full scale)'].round(6)
tbl

---
## 2 · Off-chip traffic: memory layout and casting scheme

This is the direct evidence for the **layout** and **casting** knobs: how many AXI read requests, how many beats (data chunks), and how many cycles each setting produces, measured on the real AXI port during an RTL run.

**Table columns:** `axis` says which knob this row varies (layout or casting, holding the other fixed); `setting` is the value under test; `axi_ar_requests` is how many separate read-address handshakes the hardware issued; `axi_beats` is the total data volume moved (one beat = one data chunk delivered); `total_cycles` is the whole-layer runtime. All rows are `measured (RTL)`.

In [ ]:
f4 = pd.read_csv(GC_FIG / 'f4_data_delivery_traffic.csv')
source_tag('Off-chip traffic by layout and by casting scheme', 'measured')
f4


**Reading the chart below:** the left panel holds *casting* fixed and varies *layout* — notice AR requests and cycles change a lot (channel-major coalesces into far fewer, larger bursts) while the caption's beat count does not, because layout does not change how much data crosses the chip boundary here, only how it's packaged. The right panel holds *layout* fixed and varies *casting* — here the total data volume itself changes by up to 11x, because casting controls how many times a shared value gets re-fetched.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lay = f4[f4['axis'] == 'layout']
ax = axes[0]
x = np.arange(len(lay))
ax.bar(x - 0.18, lay['axi_ar_requests'], width=0.36, color=MEASURED, label='AR requests')
ax2 = ax.twinx()
ax2.bar(x + 0.18, lay['total_cycles'], width=0.36, color='#c9c4b8', label='Total cycles')
ax.set_xticks(x); ax.set_xticklabels(lay['setting'], rotation=15)
ax.set_ylabel('AXI AR requests', color=MEASURED)
ax2.set_ylabel('Total cycles', color='#6b675c')
ax.set_title(f'Layout axis -- tiny L0, {int(lay["axi_beats"].iloc[0])} beats moved in every case\n(identical data volume; only request count / cycles differ)')

cast = f4[f4['axis'] == 'casting']
cast_tiny = cast[cast['layer'] == 'tiny L0']
ax = axes[1]
x = np.arange(len(cast_tiny))
ax.bar(x, cast_tiny['axi_beats'], color=[MEASURED, '#c9903a', '#b0463f'])
ax.set_xticks(x); ax.set_xticklabels(cast_tiny['setting'])
ax.set_ylabel('AXI beats (off-chip data volume)')
ax.set_title('Casting axis -- tiny L0, whole layer\n(same array, same layout -- volume changes by up to 11x)')
for i, v in enumerate(cast_tiny['axi_beats']):
    ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=9)

fig.tight_layout()
plt.show()


### Proof that the analytical formula reproduces this measured traffic exactly

The chooser's off-chip traffic model isn't a rough estimate — its casting-traffic formula was checked against every measured casting run this project recorded, and matches to the last beat.

In [ ]:
anchors = pd.read_csv(CHOOSER / 'eval_anchor_beats.csv')
anchors = anchors[anchors['anchor'].str.contains('beats')]
anchors = anchors.assign(delta=anchors['measured_value'] - anchors['model_value'])
source_tag('Analytical-formula prediction vs. measured RTL beat count', 'model')
anchors_display = anchors[['anchor', 'model_value', 'measured_value', 'delta', 'exact_match']].rename(
    columns={'model_value': 'model (formula)', 'measured_value': 'measured (RTL)',
             'delta': 'measured - model', 'exact_match': 'exact match'})
anchors_display


In [ ]:
# Grouped bars would be visually indistinguishable here (the two series are
# identical), which hides the finding rather than showing it -- so the delta
# is plotted directly instead, on its own axis, with the exact value labelled.
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(anchors))
ax.bar(x, anchors['delta'], color=MEASURED)
ax.set_xticks(x)
ax.set_xticklabels([a.replace(' beats', '').replace('(8x8)', '\n(8x8)') for a in anchors['anchor']], fontsize=7.5)
ax.set_ylabel('measured beats  -  model beats')
ax.set_ylim(-5, 5)
for xi, d in zip(x, anchors['delta']):
    ax.text(xi, 0.3, str(int(d)), ha='center', fontsize=9, fontweight='bold')
n_match = int(anchors['exact_match'].sum())
ax.axhline(0, color='#3a3a38', linewidth=1)
ax.set_title(f'Residual: measured minus model, all {len(anchors)} traffic anchors -- {n_match}/{len(anchors)} are exactly 0')
fig.tight_layout()
plt.show()


### Why are model and measured *exactly* equal here, but not everywhere in this notebook?

This is worth answering directly, because it looks suspicious until the reason is clear: **traffic-volume quantities (AXI beats) and timing quantities (cycles/latency) are fundamentally different kinds of formula, and only one of them was built to be exact.**

- **Off-chip traffic (above) is a *count* of discrete events** — one beat per element fetched. The analytical formula was built by mirroring the RTL prefetcher's address-generation walk step for step (the same nested loop over channels, tiles, and rows the hardware itself executes). Two exact counts of the same discrete events, computed by the same walk order, have no source of disagreement — so an exact match isn't a coincidence or a curve fit, it is the expected outcome of a structurally faithful formula, and it was checked, not assumed: all 10 traffic anchors this project recorded come back at exactly 0 beats of difference.
- **Cycle/latency estimates (Section 4 below) are a *timing approximation*, not a discrete count.** The formula prices pipeline fill/drain and throughput at a coarse, whole-array level — it does not model per-cycle handshakes, arbitration stalls, or pipeline depth the way the RTL's finite-state machine actually does. It was never intended to be cycle-exact (the source code documents it as an "upper-bound conservative" estimate), so a real, measured gap is expected there — and Section 4 shows that gap honestly (11–14%) rather than hiding it.

In short: **exact matches show up where the model was built to mirror the hardware's logic precisely; a real, quantified gap shows up where the model deliberately trades precision for a fast estimate.** Both are correct behaviour for what each formula claims to be.

---
## 3 · Memory management: STAMP vs. PAGED, banked scratchpad

**Table columns:** `layer` identifies the workload/layer under test; `metric` names what's being measured (off-chip bytes for each memory backend, page hits/misses, or bank-conflict counts at a given bank count); `value` is the measured or derived number; `source` states which. PAGED's off-chip byte figure is *derived* (measured page-fault count × the fixed 4KB page size) because the PAGED backend has no direct DRAM byte counter of its own in this hardware revision — everything else in this table is a direct hardware counter.

In [ ]:
f5 = pd.read_csv(GC_FIG / 'f5_memory_management.csv')
source_tag('STAMP vs PAGED off-chip bytes, and bank-conflict sweep', 'measured (mostly) -- see source column')
f5


**Reading the chart below:** the left panel compares how many bytes each memory-management backend actually moves off-chip for the same layer — STAMP's delta-tracking fetches only new data, so it moves noticeably less. The right panel sweeps the number of scratchpad banks and counts real arbitration conflicts — more banks spread concurrent accesses out, so conflicts fall, reaching zero once there are enough banks that collisions become structurally impossible for this access pattern.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bytes_rows = f5[f5['metric'].str.contains('off-chip bytes')]
ax = axes[0]
layers = bytes_rows['layer'].unique()
x = np.arange(len(layers)); width = 0.35
stamp_vals = [bytes_rows[(bytes_rows['layer']==l) & (bytes_rows['metric'].str.contains('STAMP'))]['value'].iloc[0] for l in layers]
paged_vals = [bytes_rows[(bytes_rows['layer']==l) & (bytes_rows['metric'].str.contains('PAGED'))]['value'].iloc[0] for l in layers]
ax.bar(x - width/2, stamp_vals, width, color=MEASURED, label='STAMP (measured)')
ax.bar(x + width/2, paged_vals, width, color='#c9903a', label='PAGED (derived from measured page faults)')
ax.set_xticks(x); ax.set_xticklabels(layers)
ax.set_ylabel('Off-chip bytes')
ax.set_title('STAMP fetches less off-chip data than the PAGED baseline')
ax.legend()

bank_rows = f5[f5['metric'].str.contains('bank conflicts')]
ax = axes[1]
banks = [int(m.split('@ ')[1].split(' ')[0]) for m in bank_rows['metric']]
ax.plot(banks, bank_rows['value'], marker='o', color=MEASURED, linewidth=2)
for b, v in zip(banks, bank_rows['value']):
    ax.annotate(str(int(v)), (b, v), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax.set_xscale('log', base=2)
ax.set_xticks(banks); ax.set_xticklabels(banks)
ax.set_xlabel('Number of scratchpad banks')
ax.set_ylabel('Bank conflicts (measured, real STAMP traffic)')
ax.set_title('Bank conflicts fall to zero as bank count grows\n(interleaved-banking arbitration behaving as designed)')

fig.tight_layout()
plt.show()


---
## 4 · The analytical configuration-chooser: is it correct?

The chooser never runs RTL — it scores all 27 (dataflow × layout × casting) combinations with the formulas validated above, in milliseconds. Its own correctness was evaluated four ways, labelled (a)–(d).

**(a) Decision accuracy** — for every (workload, goal) pair, does the chooser's top-ranked pick match the true best found by scoring all 27 combinations independently? `match=True` for all 56 trials means the chooser's own ranking logic is internally consistent — this checks the chooser doesn't contradict itself, not that it agrees with hardware (that's part (d)).

In [ ]:
acc = pd.read_csv(CHOOSER / 'eval_decision_accuracy.csv')
source_tag(f'(a) Decision accuracy: chooser top pick vs. independent exhaustive best, {len(acc)} (workload x goal) trials', 'model')
n_match = int(acc['match'].sum())
print(f'  Match: {n_match}/{len(acc)}  ({n_match/len(acc)*100:.0f}%)')
print(f'  Optimality gap when they differ: mean {acc["optimality_gap_pct"].mean():.4f}%, max {acc["optimality_gap_pct"].max():.4f}%  -- (b)')
acc.head(8)


**(c) Speed** — how long the chooser takes vs. a lower bound for what the equivalent 27 RTL runs would cost, estimated from this project's own recorded RTL wall-clock times. `source` marks which rows are timed live in this run and which are derived from historical RTL timings.

In [ ]:
spd = pd.read_csv(CHOOSER / 'eval_speed.csv')
source_tag('(c) Speed: chooser (measured this run) vs. RTL lower bound (derived from recorded wall_seconds)', 'model + measured, mixed -- see source column')
spd


**(d) Anchor check** — the chooser's picks compared directly against the two workloads with real hardware runs. `casting_axis_matches_measured` / `dataflow_axis_matches_measured` say whether the chooser's choice on that axis matches what was actually fastest/cheapest in RTL. `measured_gap_pct_vs_hw_best` is populated only where they disagree — this is the honest, quantified size of the mismatch, not a rounded-away footnote.

In [ ]:
decisions = pd.read_csv(CHOOSER / 'eval_anchor_decisions.csv')
source_tag('(d) Anchor check: chooser rankings vs. measured RTL, on the two workloads with real hardware runs', 'model, checked against measured (RTL)')
decisions


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
axes_matched = {
    'casting': decisions['casting_axis_matches_measured'].mean() * 100,
    'dataflow': decisions['dataflow_axis_matches_measured'].mean() * 100,
}
names = list(axes_matched.keys()); vals = list(axes_matched.values())
bars = ax.bar(names, vals, color=[MEASURED, GAP])
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+1.5, f'{v:.0f}%', ha='center', fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel('Agreement with measured RTL (%)')
ax.set_title('Chooser vs. measured hardware, by axis (tiny_cnn + mnist_cnn anchors)')
fig.tight_layout()
plt.show()


### Showing the dataflow gap directly — not just as a percentage

The bar above says the dataflow axis is right 0% of the time on `--goal latency`; here is exactly what that means in cycles, computed live from the same raw RTL verdict files used everywhere else in this notebook — nothing hardcoded.

In [ ]:
# Real measured cycles, read directly from the raw RTL verdict files (not
# retyped from a report) -- OS is what the chooser picks; IS is what the
# measured hardware actually runs fastest, on both anchor workloads.
pairs = [
    ('tiny_cnn',  'tiny_cnn_layer_00_OS_CHANNEL_MAJOR_STAMP_8x8_b4_verdict.json',
                  'tiny_cnn_layer_00_IS_CHANNEL_MAJOR_STAMP_8x8_b4_verdict.json'),
    ('mnist_cnn', 'mnist_cnn_layer_00_OS_CHANNEL_MAJOR_STAMP_8x8_b4_verdict.json',
                  'mnist_cnn_layer_00_IS_CHANNEL_MAJOR_STAMP_8x8_b4_ismnist_verdict.json'),
]
rows = []
for wl, os_file, is_file in pairs:
    os_cyc = json.load(open(GC_RAW / os_file))['rtl_cycles_total']
    is_cyc = json.load(open(GC_RAW / is_file))['rtl_cycles_total']
    rows.append({'workload': wl, 'chooser pick (OS) -- measured cycles': os_cyc,
                 'measured-fastest (IS) -- measured cycles': is_cyc,
                 'gap_pct': (os_cyc - is_cyc) / is_cyc * 100})
gap_df = pd.DataFrame(rows)
source_tag('Real measured cycles behind the dataflow-axis gap, read live from raw RTL verdict files', 'measured')
gap_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(gap_df)); width = 0.35
ax.bar(x - width/2, gap_df['chooser pick (OS) -- measured cycles'], width, color=MODEL, label='OS (chooser pick), measured cycles')
ax.bar(x + width/2, gap_df['measured-fastest (IS) -- measured cycles'], width, color=MEASURED, label='IS (measured-fastest), measured cycles')
ax.set_xticks(x); ax.set_xticklabels(gap_df['workload'])
ax.set_ylabel('Cycles (measured RTL)')
for xi, row in gap_df.iterrows():
    ax.text(xi, max(row.iloc[1], row.iloc[2]) * 1.02, f"+{row['gap_pct']:.1f}%", ha='center', fontweight='bold', color=GAP)
ax.set_title('The dataflow-axis gap, in real measured cycles -- both bars are real RTL runs;\nthe chooser picks the left bar, hardware is actually faster on the right')
ax.legend()
fig.tight_layout()
plt.show()
print('\nThis is a genuinely open item, documented rather than hidden: the cycle-estimator formula')
print('resolves the OS-vs-IS difference at only ~0.09%% internally, while measured hardware runs')
print('11-14%% apart. It is exactly the kind of gap the explanation in Section 2 predicts: a timing')
print('APPROXIMATION, not a structural mirror of the RTL, so it is not expected to be exact.')


---
## 5 · The edge/cloud experiment suite — what it is, and what it is *not*

This project also contains a much larger sweep across 14 industry-scale reference DNNs (MobileNetV2, ResNet, BERT, DLRM, GPT-2, etc. — `results/edge/` and `results/cloud/`). **It is important to be precise about this suite in an evaluation: none of it is RTL-measured.** It is a pure analytical design-space exploration, generated by `scripts/run_full_eval.py`, using hand-authored cost formulas — the same family of formulas validated above, but applied to workloads that were never run through Verilator (they are far too large to simulate in RTL in reasonable time).

In [ ]:
exec_cyc = pd.read_csv(EDGE_CLOUD / 'edge_exp7_execution_cycles.csv')
source_tag('CAUTION -- despite the column name, this is NOT measured RTL data', 'model (synthetic)')
print("scripts/run_full_eval.py computes the 'rtl_actual' column as:")
print("    rtl_actual = python_estimated_cycles * a FIXED assumed multiplier (1.03-1.05, one per dataflow)")
print('That multiplier is an assumption written into the script, not something measured on this workload.')
exec_cyc.head(6)


In [ ]:
fpga = pd.read_csv(EDGE_CLOUD / 'edge_exp7_fpga_resources.csv')
source_tag('CAUTION -- FPGA utilisation numbers computed from a hand-authored formula, no synthesis tool run', 'model (synthetic)')
print('lut_util/dsp_util/bram_util/op_freq_norm come from a power-law formula (e.g. 22 * (array/8)**1.8),')
print('picked to look plausible -- no Vivado, no synthesis, no FPGA was ever involved in producing these.')
fpga


**How to describe this suite correctly in the evaluation, if asked:**
> "The edge/cloud suite is a design-space exploration tool: it applies the same cost-model formulas we validated against real RTL on tiny_cnn and mnist_cnn to a much larger set of reference DNNs, to show how trends generalize. It is explicitly labelled `model`, not `measured`, everywhere it appears in our reports — including the `exp7_hw_verification` file, whose `rtl_actual` and FPGA-utilisation columns are computed from assumed correction factors and a synthetic formula, not from a real hardware or FPGA run."

---
## 6 · Process note: how this was actually built and verified

A fair question in an evaluation is: *did you personally run Verilator, cocotb, and TensorFlow yourself and collect this data — or did an AI just generate these numbers?* Both halves of that question have a real answer, and the honest one is also the strongest one to give.

**The numbers are real.** Every value in this notebook is a reproducible output of **Verilator** (RTL simulation), **cocotb** (the testbench driver), and **TensorFlow** (the golden reference) — never invented, never approximated by an AI in place of running the tool.

**An AI-assisted development environment (Claude Code) was used throughout implementation and verification** — to help write and debug the RTL, author the cocotb testbenches, and execute the verification commands. This is the same role a build script, a Makefile, or an IDE assistant plays in any modern engineering workflow: it orchestrates and accelerates real tools, it does not replace them.

**The actual pipeline, step by step:**
1. RTL (SystemVerilog) and cocotb testbenches were written and iterated to implement and exercise the stationary schemes, memory layouts, and casting schemes.
2. For every configuration, the same two-command pipeline ran: Verilator compiles the RTL into a cycle-accurate simulator; cocotb drives it with real layer data and records every counter (cycles, AXI beats, AR requests, correctness).
3. Each run wrote a real result file — the exact JSON files loaded throughout this notebook — deterministically: re-running the same command against the same RTL reproduces the same file.
4. The tables and charts in this notebook were generated by reading those files with pandas/matplotlib. No number here was typed in by hand.

**If asked directly, this is accurate to say:**
> "I used Claude Code, an AI-assisted development environment, to help implement and verify this design — including writing RTL fixes, authoring testbenches, and running the verification suite. Every number shown comes from a real Verilator simulation checked against a TensorFlow golden reference; none of it was generated by the AI. It's fully reproducible: the exact command behind any result is on record, and re-running it gives the same output, because it's a deterministic simulation of real hardware logic — not a language model's guess."

The strongest evidence for this in the room: **offer to re-run any single command live.** Because the pipeline is deterministic tooling, not a black box, that offer is safe to make.

---
## 7 · Summary for the thesis / evaluation

| Question | Answer, in one line |
|---|---|
| Is the RTL functionally correct? | Yes — 26/26 measured configurations pass within 5% tolerance; worst case is 100x inside the band and consistent with fixed-point rounding, not a bug. |
| Is the off-chip traffic model validated? | Yes, exactly — the casting-traffic formula reproduces all 6 measured beat counts exactly; the analogous WS re-fetch formula reproduces all 4 measured WS runs exactly. It matches exactly because it mirrors the RTL's own address-generation logic step for step. |
| Is the chooser's latency estimate exact too? | No, and it isn't supposed to be — it's a coarse timing approximation, not a structural mirror. Measured gap: 11–14% on the two hardware-anchored workloads, shown directly in Section 4, not rounded away. |
| Is the chooser's decision-making correct? | Its top pick matches exhaustive search 100% of the time (by construction, since it *is* exhaustive over 27 combos); on the two hardware-anchored workloads, it agrees with measured RTL on the casting and layout axes, and is honestly documented as under-resolving the dataflow-latency axis. |
| Are the edge/cloud numbers measured? | No — clearly labelled `model`, included to show the same formulas generalize, not as hardware evidence. |

### Exporting for a LaTeX thesis
Any table above can be exported directly, e.g.:

In [ ]:
# Export the tables for the thesis. `show` is the correctness table with
# measured/golden/error in separate columns; `tbl` is the stationary x
# layout x casting performance table.
show.to_csv('correctness_detail_export.csv', index=False)
tbl.to_csv('config_axes_performance_export.csv', index=False)
print('Wrote correctness_detail_export.csv and config_axes_performance_export.csv')
print('-- download them from the Colab file browser (folder icon, left side).')
try:
    # to_latex() needs the optional 'jinja2' package; not guaranteed present.
    print('\nLaTeX table (requires jinja2):\n')
    print(show.to_latex(index=False)[:600], '...')
except ImportError:
    print("\n(LaTeX export needs 'jinja2' -- run `!pip install jinja2` above and re-run.)")
